In [1]:
import os
os.chdir("d:/Kidney_classification_Using_MLOPS_and_DVC_Data-version-control")
print("Working directory:", os.getcwd())

Working directory: d:\Kidney_classification_Using_MLOPS_and_DVC_Data-version-control


In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_learning_rate: float

In [3]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        training_data = Path(self.config.data_ingestion.unzip_dir) / "kidney-ct-scan-image"
        create_directories([training.root_dir])
        return TrainingConfig(
            root_dir=training.root_dir,
            trained_model_path=training.trained_model_path,
            updated_base_model_path=prepare_base_model.updated_base_model_path,
            training_data=training_data,
            params_epochs=self.params.EPOCHS,
            params_batch_size=self.params.BATCH_SIZE,
            params_is_augmentation=self.params.AUGMENTATION,
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
        )

In [5]:
import os
import tensorflow as tf
from pathlib import Path


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path, compile=False
        )
        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=self.config.params_learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

    def train_valid_generator(self):
        datagenerator_kwargs = dict(rescale=1.0 / 255, validation_split=0.20)
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )
        self.save_model(path=self.config.trained_model_path, model=self.model)

In [6]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise e

[2026-03-01 17:05:41,126: INFO: common]: YAML file loaded successfully: config\config.yaml
[2026-03-01 17:05:41,130: INFO: common]: YAML file loaded successfully: params.yaml
[2026-03-01 17:05:41,132: INFO: common]: Created directory: artifacts
[2026-03-01 17:05:41,133: INFO: common]: Created directory: artifacts/training
Found 93 images belonging to 2 classes.
Found 372 images belonging to 2 classes.
Epoch 1/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.5112 - loss: 12.8361 - val_accuracy: 0.4375 - val_loss: 8.4780
Epoch 2/5
 1/23 ━━━━━━━━━━━━━━━━━━━━ 1:12 3s/step - accuracy: 0.8750 - loss: 2.2594

c:\Users\sento\anaconda3\envs\kidney\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 446ms/step - accuracy: 0.8750 - loss: 2.2594 - val_accuracy: 0.4375 - val_loss: 16.6194
Epoch 3/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step - accuracy: 0.7753 - loss: 3.7327 - val_accuracy: 0.4375 - val_loss: 3.6431
Epoch 4/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 11s 420ms/step - accuracy: 0.8750 - loss: 1.2208 - val_accuracy: 0.4750 - val_loss: 12.8845
Epoch 5/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.7107 - loss: 5.6108 - val_accuracy: 0.6000 - val_loss: 11.7546
[2026-03-01 17:08:41,468: WARNING: saving_api]: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
